# Lab 12 — Model Selection & Metrics — Credit Default
**Evaluation Track** · Intermediate · ~60 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Compute majority-class baseline accuracy
2. Plot ROC and PR curves for 3 models
3. Use k-fold CV for fair comparison
4. Sketch a reliability diagram for calibration

## Datasets (this folder)
- `Default.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-02-classical-machine-learning/labs/lab-12-model-selection-default/lab-12-model-selection-default.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 2).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`Default.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-12-model-selection-default"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-02-classical-machine-learning/labs/lab-12-model-selection-default/bundle/dataset.zip"
NEED = ["Default.csv"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Model Selection: Credit Default — Accuracy Is a Trap

> **Scenario:** A bank's risk team must flag likely credit-card defaulters. Only ~3% of customers default, so a model that says "no default" for everyone scores 97% accuracy — and is worthless. You will learn why accuracy misleads on imbalanced data, how ROC and PR curves tell the real story, and how to compare Logistic Regression vs Random Forest vs Gradient Boosting honestly.
>
> **You will learn:** the accuracy paradox, class imbalance, ROC-AUC vs PR-AUC, stratified k-fold CV, probability calibration, reliability diagrams.
> **Time:** ~60 minutes. **Level:** Intermediate. **Needs:** pandas + sklearn. **Env:** 🟢 Colab only.

### Mental Map

| Section | Key idea | sklearn / pandas tool |
|---|---|---|
| 1 | Encode the target; feel the imbalance | `df.map` |
| 2 | The accuracy paradox | `accuracy_score` |
| 3 | ROC and PR curves | `roc_curve`, `precision_recall_curve` |
| 4 | Three models, honest comparison | `cross_val_score`, `StratifiedKFold` |
| 5 | Calibration: do probabilities mean anything? | `calibration_curve` |

---

## 1. Setup: Encode and Feel the Imbalance

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv("Default.csv")
df = df.drop(columns=["rownames"])
df["default"] = df["default"].map({"No": 0, "Yes": 1})
df["student"] = df["student"].map({"No": 0, "Yes": 1})

print(df.shape)
print(df["default"].value_counts())
print(f"default rate: {df['default'].mean():.4f}")


**What to notice:**
1. 10,000 rows, 333 defaulters → a 3.3% positive rate.
2. `df.map` (not the removed `df.applymap`) encodes Yes/No to 1/0 in one line.
3. Every metric you choose must respect this imbalance — accuracy does not.

> **Pitfall:** `df.applymap` was removed in pandas 2.1. Use `df.map` on DataFrames or `Series.map` on single columns.

---

## 2. The Accuracy Paradox

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

X = df[["student", "balance", "income"]]
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

majority_pred = np.zeros_like(y_test)
print(f"majority-class baseline accuracy: {accuracy_score(y_test, majority_pred):.4f}")
print(f"test default rate: {y_test.mean():.4f}")


**What to notice:**
1. Predicting "no default" for everyone gives 0.9668 accuracy — higher than many real models!
2. This is the **accuracy paradox**: on imbalanced data, accuracy rewards ignoring the minority class.
3. The fix is a metric that focuses on the positive class: precision, recall, F1, PR-AUC.

> **Pitfall:** never report accuracy alone on imbalanced data. Always pair it with a confusion-matrix-derived metric.

---

## 3. ROC and PR Curves

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, precision_recall_curve,
                             roc_auc_score, roc_curve)

models = {
    "Logistic": LogisticRegression(max_iter=1000, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
}

probs = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    probs[name] = model.predict_proba(X_test)[:, 1]
    print(f"{name:16s} ROC-AUC: {roc_auc_score(y_test, probs[name]):.4f}   "
          f"PR-AUC: {average_precision_score(y_test, probs[name]):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, p in probs.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    axes[0].plot(fpr, tpr, label=name)
    prec, rec, _ = precision_recall_curve(y_test, p)
    axes[1].plot(rec, prec, label=name)
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_title("ROC curve"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[1].axhline(y_test.mean(), color="k", ls="--", lw=1, label="baseline")
axes[1].set_title("PR curve"); axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
for ax in axes: ax.legend()
fig.tight_layout()
fig.savefig("lab12_roc_pr.png", dpi=120)
from IPython.display import display
display(fig)
print("saved lab12_roc_pr.png")


**What to notice:**
1. ROC-AUC looks rosy (0.89–0.95) because the huge true-negative mass inflates it; PR-AUC spreads the models honestly (0.35–0.51).
2. The PR baseline is the positive rate (0.033) — a model must beat that horizontal line to be useful.
3. Logistic regression wins on **both** metrics: default is driven almost linearly by `balance`, so the fancy ensembles just add variance.

> **Pitfall:** with 3% positives, a single threshold change can swing precision wildly. Always look at the whole curve, not one operating point.

---

## 4. Honest Comparison with Stratified k-Fold CV

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring="average_precision", n_jobs=-1)
    print(f"{name:16s} CV PR-AUC: {scores.mean():.4f} +/- {scores.std():.4f}")


**What to notice:**
1. `StratifiedKFold` keeps the 3.3% positive rate in every fold — plain KFold could produce a fold with almost no positives.
2. `average_precision` (PR-AUC) is the right scoring string for imbalanced binary problems.
3. CV on all 10,000 rows uses more data than a single 7,500-row train split, so these numbers are the ones to trust. Logistic still leads (0.535).

> **Pitfall:** `cross_val_score` clones the models, so reusing the fitted `models` dict here is safe — the fitted copies are untouched.

---

## 5. Calibration: Do the Probabilities Mean Anything?

In [ ]:
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(6, 5))
for name, p in probs.items():
    frac_pos, mean_pred = calibration_curve(y_test, p, n_bins=10)
    ax.plot(mean_pred, frac_pos, marker="o", label=name)
ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Reliability diagram")
ax.legend()
fig.tight_layout()
fig.savefig("lab12_reliability.png", dpi=120)
from IPython.display import display
display(fig)
print("saved lab12_reliability.png")


**What to notice:**
1. Logistic regression hugs the diagonal in the high-probability bins — its outputs can be read as real risks.
2. Tree ensembles tend to push probabilities toward 0 and 1 (overconfident), so their curves bow away from the diagonal.
3. If you need true probabilities (e.g., expected-loss pricing), wrap the model in `CalibratedClassifierCV` with `method="isotonic"` or `"sigmoid"`.

> **Pitfall:** calibration and discrimination are different axes. A model can rank well (high AUC) and still be miscalibrated.

---

## Exercises (do these!)

### Exercise 1 — Majority-class baseline
Compute the accuracy of always predicting "no default" on the test set.
*Expected: ≈ 0.9668.*

**Follow-up:** What is the test-set default rate? Check: ≈ 0.0332.

### Exercise 2 — PR-AUC for all three models
Report PR-AUC for Logistic, RandomForest, and GradientBoosting on the test set.
*Expected: Logistic ≈ 0.509, RandomForest ≈ 0.386, GradientBoosting ≈ 0.354.*

**Follow-up:** Which model wins on ROC-AUC? Check: Logistic (≈ 0.945).

### Exercise 3 — Reliability diagram
Sketch the reliability diagram and report the logistic model's mean predicted probability vs actual fraction of positives in the highest bin.
*Expected: mean predicted ≈ 0.95, fraction ≈ 1.00.*

**Follow-up:** Is logistic regression close to the diagonal in the high bins? Check: yes — max deviation ≈ 0.05 for bins above 0.5.

---

## Solutions

Try for 15 min each before peeking.

In [ ]:
# --- Solution 1 ---
base_acc = accuracy_score(y_test, np.zeros_like(y_test))
print(f"majority baseline accuracy: {base_acc:.4f}")
print(f"test default rate: {y_test.mean():.4f}")

# --- Solution 2 ---
for name, p in probs.items():
    print(f"{name:16s} PR-AUC: {average_precision_score(y_test, p):.4f}  "
          f"ROC-AUC: {roc_auc_score(y_test, p):.4f}")

# --- Solution 3 ---
frac_pos, mean_pred = calibration_curve(y_test, probs["Logistic"], n_bins=10)
print("logistic reliability (mean_pred vs frac_pos):")
for mp, fp in zip(mean_pred, frac_pos):
    print(f"  {mp:.3f} vs {fp:.3f}")

# --- Follow-up 1 ---
assert abs(base_acc - (1 - y_test.mean())) < 1e-9
print("baseline == 1 - default rate: OK")

# --- Follow-up 2 ---
pa = {n: average_precision_score(y_test, p) for n, p in probs.items()}
best_pa = max(pa, key=pa.get)
print(f"best PR-AUC: {best_pa} ({pa[best_pa]:.4f})")

# --- Follow-up 3 ---
high = mean_pred > 0.5
dev = np.abs(frac_pos[high] - mean_pred[high])
print(f"max |calibration deviation| above 0.5: {dev.max():.3f}")
assert dev.max() < 0.1, "logistic should be well calibrated in the high bins"


### What to learn next
- Try `class_weight="balanced"` on the logistic model and watch recall rise while precision falls.
- Plot a precision–recall trade-off curve and pick a threshold for a target recall.
- Explore `CalibratedClassifierCV` to fix the tree models' overconfident probabilities.

*Files in this folder: Default.csv.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
